In [38]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch9. RAG</font>

## RAG 구현 절차
1. docx 문서 로드 (python-docx : \n를 제외하고 읽어옴)
2. 문서 분할 (text_spliter 사용)
    - 모델의 context window와 token 비용 확인
    - chunk(분할된 문서 단위, 클수록 비용과 시간 소요 증가)로 분할
3. 분할된 문서 임베딩 → vector Store(DB)에 저장 ; Chroma(local)
4. 입력 받은 질문과 vector Store에 저장된 문서의 유사도 검색
5. 유사도 상위 n개의 문서 chunk를 LLM에 질문과 함께 전달하여 답변 생성

# 1. 문서 로드

In [7]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

CPU times: total: 6.09 s
Wall time: 6.49 s


1

# 2. 분할하며 로드


## 2.1 TokenTextSplitter
- 1500 token 단위로 겹쳐서 분할
```
 1. "o200k_base" :
    - 주요 모델: gpt-4o, gpt-4o-mini 등 최신 세대 모델
    - 설명: 최신 모델에 최적화된 인코딩 방식

 2. "cl100k_base" :
    - 주요 모델: gpt-4, gpt-3.5-turbo, text-embedding-3 등
    - 설명: ChatGPT 및 임베딩 모델에서 가장 널리 쓰이는 표준 인코딩

 3. "p50k_base" :
    - 주요 모델: text-davinci-003, Codex 등
    - 설명: 구버전 다빈치 모델 및 코드 처리에 최적화된 인코딩

 4. "r50k_base" (또는 "gpt2") :
    - 주요 모델: GPT-2, 구버전 GPT-3 모델
    - 설명: 가장 오래된 레거시 호환용 인코딩
```

In [10]:
%%time
from langchain_text_splitters import TokenTextSplitter
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=1500,
    chunk_overlap=200
)

documents = loader.load_and_split(text_splitter=text_splitter)

CPU times: total: 7.64 s
Wall time: 9.34 s


## 2.2 RecursiveCharacterTextSplitter
- 1500자 단위로 겹쳐서 분할

In [2]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Admin\AppData\Local\Temp\ipykernel_15260\3095330809.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [3]:
loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n\n\n", "\n\n", "\n", " "] # 분할 기준 우선순위
)

documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print(runtime)

13.834391355514526


# 3. 분할된 문서 임베딩 → vector Store(DB)에 저장

In [9]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

import os
load_dotenv()

embedding = UpstageEmbeddings(
    model='solar-embedding-2-query'
)

In [10]:
len(embedding.embed_query("소득세법의 예시 구문"))

1024

In [11]:
vectors = embedding.embed_documents([document.page_content for document in documents])

In [12]:
len(vectors[0]), len(vectors[1])

(1024, 1024)

In [14]:
%%time
from langchain_chroma import Chroma

# 데이터 최초 저장 시
database = Chroma.from_documents(
    documents=documents, # 분할된 documents
    embedding=embedding, # 임베딩 객체
    collection_name='tax_collection', # 생략 시 랜덤 생성
    persist_directory='./chroma_upstage' # 생략 시 local file 저장 불가 (인메모리 변수 저장 처리)
)

CPU times: total: 3.12 s
Wall time: 7.68 s


In [18]:
%%time

# 데이터 로드 시
database = Chroma(
    embedding_function=embedding,
    collection_name='tax_collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 46.9 ms
Wall time: 29.2 ms


In [19]:
results = database.get(include=['metadatas', 'embeddings', 'documents'])
print(f'데이터 수 : {len(results["ids"])}')
print(f'문서 임베딩 차원 수 : {len(results["embeddings"][0])}')

데이터 수 : 234
문서 임베딩 차원 수 : 1024


# 4. Retrieval 작업 (유사도 검색)

In [20]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(
    query=query, 
    k=2 # 기본값 : 4
)

In [21]:
len(retrieved_docs)

2

In [22]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 5. 질문과 doc을 LLM에 전달 및 답변 생성

In [25]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model='solar-pro4')

In [26]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가 입니다
- 다음의 [context]를 참조하여 사용자의 질문에 답변하세요
[context]
{retrieved_doc}
질문 : {query}"""

ai_msg = llm.invoke(prompt)

In [27]:
print(ai_msg.usage_metadata)

{'input_tokens': 1656, 'output_tokens': 918, 'total_tokens': 2574, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [28]:
from langchain_core.prompts import ChatPromptTemplate

promptTemplate = ChatPromptTemplate([
    ('system','당신은 최고의 한국 소득세법 전문가입니다'),
    ('human',f'''다음 문맥을 참고하여 5문장 이내로 축약하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    질문 : {{question}}, 문맥 : {{context}}, 답변 : ''')
])

In [29]:
prompt = promptTemplate.invoke(
    {
    'context':retrieved_doc,
    'question':query
    }
)

In [30]:
ai_message = llm.invoke(prompt)

In [31]:
# StrOutputParser 활용
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'제공된 문맥만으로는 연봉 5천만원인 직장인의 소득세를 정확히 계산할 수 없습니다. 소득세는 근로소득공제, 인적공제, 연금보험료 공제 등 다양한 공제 항목을 적용한 종합소득과세표준에 기본세율(6~45%)을 적용하여 산출되기 때문입니다. 제공된 맥락은 근로소득세액공제, 자녀세액공제, 근로소득공제 등의 구체적 수준과 계산 공식을 규정하고 있으나, 세율을 포함한 과세표준 산출의 전체 과정은 담고 있지 않습니다. 따라서 기본적으로 총급여 5천만원에서 근로소득공제(약 1,200만원)를 차감한 금액이 과세표준의 기초가 되나, 최종 소득세액은 다른 공제 사항과 적용 세율에 따라 결정됩니다. 정확한 세액을 알기 위해서는 개인의 공제 항목과 해당 연도의 세율을 적용하여 계산해야 합니다.'

In [32]:
# 한번에
outputParser.invoke(llm.invoke(promptTemplate.invoke({
    'context':retrieved_doc,'question':query
    })))

'제공된 문맥은 근로소득공제와 근로소득세액공제, 자녀세액공제 등의 공제 요건만을 규정하고 있으며, 과세표준 구간별 세율과 최종 산출세액 계산 공식이 포함되어 있지 않습니다. 따라서 연봉 5천만 원에 대한 정확한 소득세 금액을 산출할 수 없습니다. 소득세는 근로소득공제(5천만 원 기준 약 1,200만 원) 적용 후 남은 과세표준에 기본세율을 적용하고, 여기서 근로소득세액공제 등을 차감하여 결정되므로, 정확한 금액은 구체적인 부양가족 수, 다른 공제 항목, 적용 세율 등을 종합적으로 고려해야 합니다. 결론적으로 제시된 문맥만으로는 해당 연봉의 소득세 금액을 알 수 없습니다.'

In [33]:
# LCEL 활용
chain = promptTemplate | llm | outputParser
chain.invoke({'context':retrieved_doc,'question':query})

'제공된 문맥만으로는 연봉 5천만원인 직장인의 정확한 소득세를 산출할 수 없으므로 답변은 불가능합니다. 제시된 조문(제47조, 제59조, 제59조의2)은 근로소득공제, 근로소득세액공제, 자녀세액공제의 공제액 계산 기준만을 규정하고 있을 뿐, 과세표준이나 세율을 적용하는 종합소득공제 및 세율표가 포함되어 있지 않기 때문입니다. 소득세를 정확히 계산하려면 총급여에서 근로소득공제와 각종 공제를 차감하여 과세표준을 도출한 후, 종합소득세율과 지방소득세를 적용하고 세액공제 한도까지 반영해야 합니다. 따라서 특정 공제 대상 여부나 다른 소득 유무 등의 추가 정보와 과세표준 산정 기준이 필요하며, 문맥만으로는 질문에 답할 수 없습니다.'

LangChain으로 전달

In [34]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [36]:
# 임베딩 & LLM 초기화
load_dotenv()
embedding = UpstageEmbeddings(model = 'solar-embedding-2-query')
llm = ChatUpstage(model = 'solar-pro4')

# vectorDB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma_upstage',
    collection_name='tax-collection'
)

# Retriever 생성
# 1 : 유사도 상위 k개
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)
# 2 : 유사도 상위 fetch_k개 중 내용이 겹치지 않는 k개
# retriever = vectorStore.as_retriever(
#     search_type='mmr',
#     search_kwargs={
#         'k':3,
#         'fetch_k':10,
#         'lambda_mult':0.5
#     }    
# )
# 3 : score_threshold 이상의 유사도 상위 k개
# retriever = vectorStore.as_retriever(
#     search_type='similarity_score_threshold',
#     search_kwargs={
#         'k':3,
#         'score_threshold':0.4
#     }
# )

# 프롬프트 템플릿
template = f'''당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 최대 5문장으로 답하세요. 답을 모르면 모른다고 답하세요.
질문 : {{query}}
문맥 : {{context}}
답변 : '''
prompt = ChatPromptTemplate.from_template(template)

# 검색한 문서 텍스트 변환
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [37]:
# RAG chain
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)

rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼만가요?')

'2024년 기준 소득세액을 추정하려면 과세표준을 정확히 계산해야 하므로 단순 연봉만으로 확정세액을 단정할 수 없습니다. 연간 총급여에서 근로소득공제, 인적공제, 연금·보험료·신용카드 등 기본·특별공제를 차감하면 과세표준이 줄어들며, 연말정산 시 적용되는 공제 항목과 가족 구성에 따라 실제 세액이 크게 달라집니다. 일반적으로 다수의 기본공제를 적용받는 1인 직장인의 경우 원천징수 또는 연말정산 기준 연간 소득세가 150만 원에서 250만 원 사이로 형성되는 사례가 많습니다. 다만 이는 개략적인 추산이며, 4대 보험료나 건강보험 산정 방식, 지방소득세(소득분의 10%)까지 포함하면 실제 부담하는 총 세 부담은 더 높아질 수 있습니다. 정확한 금액은 본인의 공제자료와 국세청 홈택스 모의계산 또는 세무 전문가 확인을 통해 확정하시기 바랍니다.'